In [ ]:
import os
import math
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
from tqdm import trange, tqdm
import matplotlib.pyplot as plt
%matplotlib inline
import torch
import torch.nn.functional as F  
from torch.utils.tensorboard import SummaryWriter

from Utils.CADTensorGenerator import CADTensorGenerator
from Utils.CADVisualizer   import CADVisualizer
from HDVClassNet.PP_net import PPNet
from HDVClassNet.VoronoiDecorder import VoronoiDecoder
from Training.MainTrain import TrainingConfig, NN_Trainer
from neuraltomo_fem import run_fem_loss
from problems.TipCantilever_30_20_20_midLoad import TipCantilever_30_20_20_midLoad
from problems.ThickenShell import ThickenShell

import pyvista as pv
try:
    pv.set_jupyter_backend("trame")
except Exception:
    pass

# ---- Reproducibility (recommended for D_params comparisons) ----
SEED = 20
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

BASE = Path(__file__).parent if "__file__" in globals() else Path.cwd()
print("Code Directory:", BASE)
TesPartsDir = BASE / "Testparts" 
print("Test Step files Directory:", TesPartsDir)
# ---- Device ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


Code Directory: /home/arash/VoronoiImp-main
Test Step files Directory: /home/arash/VoronoiImp-main/Testparts
device: cuda


In [2]:
viz = CADVisualizer()
# Laoding model and extracting mesh and tensors as input
FreeFormSurf1  = TesPartsDir / "FreeFormCrv1.stp"
FreeFormSurf2A = TesPartsDir / "FreeFormSurf2A.STEP"
YachtBodypart  = TesPartsDir / "YachtBodypart.stp"
CircularSurf1  = TesPartsDir / "CircularSurf1.stp"
Cube           = TesPartsDir / "Cube.stp"
CircularSur2   = TesPartsDir / "CircularSur2.stp"
Conic          = TesPartsDir / "Conic.stp"
CircularHoles  = TesPartsDir / "CircularHoles.stp"
FullCylinder   = TesPartsDir / "FullCylinder.stp"
Sphere         = TesPartsDir / "Sphere.stp"
SphereTap      = TesPartsDir / "SphereTap.stp"
Tidebottle     = TesPartsDir / "Tidebottle.STEP"

shape_path = str(CircularSurf1)
generator = CADTensorGenerator(
    deflection=0.1,
    angle=0.5,
    metric_tol=1e-9,
    det_min=1e-5,
    n_u=100,
    n_v=100,
    device=device,
)
mesh_df, faces_df, tensors = generator.generate_from_file(shape_path, input_ring=1)

# mesh_df, faces_df, tensors = generator.generate_from_file(
#     shape_path=shape_path,
#     input_ring=1,
#     visualize=True,
# )
print(tensors["num_faces"])
print(tensors["face_tensors"][0].keys())
print(tensors["face_periodicity"])
uv = tensors["uv"]
points_xyz = tensors["points_xyz"]
uv = tensors["uv"]
Xu = tensors["Xu"]
Xv = tensors["Xv"]
points_xyz = tensors["points_xyz"]
face_areas = tensors["face_areas"]
faces_ijk = tensors["faces_ijk"]
face_id = tensors["face_id"]
boundary_idx_ring1 = tensors["boundary_idx_ring1"]
pv_faces = tensors["pv_faces"]
viz.visualize_show_Model(points_xyz, pv_faces)
tensors["BBX"]
print("BBX:", tensors["BBX"])

MinVolFrac: 0.059099987149238586
uv device: cuda:0
1
dict_keys(['face_id', 'uv', 'points_xyz', 'Xu', 'Xv', 'faces_ijk', 'pv_faces', 'face_areas', 'boundary_idx', 'boundary_idx_ring1', 'boundary_idx_ring2', 'min_vol_frac', 'BBX', 'u_periodic', 'v_periodic', 'u_period', 'v_period', 'u_raw_bounds', 'v_raw_bounds', 'global_vertex_idx', 'global_face_idx', 'num_vertices', 'num_faces'])
{0: {'u_periodic': False, 'v_periodic': False, 'u_period': 6.283185307179586, 'v_period': None}}


Widget(value='<iframe src="http://localhost:40881/index.html?ui=P_0x7900a19c9a20_0&reconnect=auto" class="pyvi…

BBX: {0: {'xmin': 0.0, 'xmax': 10.0, 'ymin': -0.12739957269417435, 'ymax': 2.1009186367176094, 'zmin': -2.032529852280162, 'zmax': 1.9386755085904284}}


In [3]:
fixed_height_shell= 0.5
shell_problem = ThickenShell(
    thickness=fixed_height_shell,
    BC_dir = "x",
    Load_magnitude=1.0,
    voxel_size=0.25,
    extra_layers=1,
    tensors=tensors,
    tangential_tol=0.1,
)

fem = run_fem_loss.NeuralTOMOFEM(shell_problem, device=device, isotropic=False)
shell_problem.debug_voxel_stats()
shell_problem.show_voxels_surface_and_bc_NEW()

=== Voxel Stats ===
brep_bbox: {'xmin': 0.0, 'xmax': 10.0, 'ymin': -0.12739957269417435, 'ymax': 2.1009186367176094, 'zmin': -2.032529852280162, 'zmax': 1.9386755085904284}
mesh: {'nelx': 44, 'nely': 13, 'nelz': 20, 'elemSize': array([0.25, 0.25, 0.25]), 'type': 'grid'}
elem_centers shape: (20, 44, 13, 3)
node_coords shape: (21, 45, 14, 3)
occupied voxels: 2200
total voxels: 11440
occupancy ratio: 0.19230769230769232
voxelized volume: 34.375
thickness: 0.5
voxel_size: 0.25
target approx volume (sum(face_areas)*thickness): 32.80144500732422
volume ratio voxel/target: 1.0479721241647868


Widget(value='<iframe src="http://localhost:40881/index.html?ui=P_0x79032409e020_1&reconnect=auto" class="pyvi…

In [ ]:
cfg = TrainingConfig(
    seed_number=10,
    use_anisotropy=False,
    fixed_height=fixed_height_shell,
    target_volfrac=0.2,
    seed_repulsion_sigma=0.08,
    boundary_margin=0.05,
    w_min=0.0005,
    w_max=0.5,

    lam_fem=10.0,
    lam_vol=1.0,
    lam_rep=0.2,
    lam_bnd=0.01,

    # differentiable strutness loss
    lam_strut=0.02,
    lam_strut_edge=1.0,
    lam_strut_void=0.2,

    lr_seed_refine=2e-3,
    lr_delta_head=2e-4,
    lr_mlp=2e-4,
    lr_w_head=2e-4,
    lr_h_head=2e-4,

    comp_normalize_by=1e10,
    normalize_losses=False,

    fem_density_floor=0.02,
    skip_bad_fem_steps=True,

    num_steps=1000,
    context_vector_size=8,

    # tau controls soft Voronoi ownership
    tau=0.1,

    # beta controls sharpness of the geometric bisector band
    beta=0.02,

    log_every=50,
    early_stop_start=500,
    patience=500,
    min_delta=1e-8,
    experiment_name="case1_half_cylinder",
    tensorboard_enabled=True,
    tb_log_histograms_every=100,
)
trainer = NN_Trainer(
    generator=generator,
    viz=viz,
    decoder_cls=VoronoiDecoder,
    ppnet_cls=PPNet,
    fem=fem,
    shell_problem=shell_problem,
    config=cfg,
)

result = trainer.train(
    uv=uv,
    Xu=Xu,
    Xv=Xv,
    points_xyz=points_xyz,
    face_areas=face_areas,
    faces_ijk=faces_ijk,
    face_id=face_id,
    boundary_idx_ring1=boundary_idx_ring1,
    face_tensors=tensors["face_tensors"],
)

trainer.visualize_result_stepwise(result, points_xyz, faces_ijk)
trainer.visualize_result_final(result, points_xyz, faces_ijk, thr=0.5, show_solid=False)

[00000] L_total=2.1296e-02 | L_vol=2.313e-03 L_fem=2.955e-06 L_strut=7.018e-01 L_rep=1.299e-06 L_bnd=4.918e-01 | vol=0.258 vol_eff=0.248 (/0.200) comp=2.955e+04 w=2.515e-01 | Lse=7.018e-01 Lsv=0.000e+00 rho(min/mean/max)=0.001/0.261/0.902 | rho_b=0.168 rho_v=0.207 | Δrho=0.00e+00 Δseed=0.00e+00 grad_mean=1.92e-04 | fem=OK | best=2.1296e-02@0
[00050] L_total=2.0500e-02 | L_vol=2.214e-03 L_fem=4.472e-06 L_strut=6.925e-01 L_rep=9.079e-06 L_bnd=4.390e-01 | vol=0.262 vol_eff=0.247 (/0.200) comp=4.472e+04 w=2.488e-01 | Lse=6.925e-01 Lsv=0.000e+00 rho(min/mean/max)=0.000/0.265/0.913 | rho_b=0.168 rho_v=0.214 | Δrho=2.38e-01 Δseed=6.56e-02 grad_mean=2.00e-04 | fem=OK | best=2.0463e-02@49
[00100] L_total=2.0112e-02 | L_vol=1.929e-03 L_fem=3.655e-06 L_strut=6.966e-01 L_rep=1.670e-06 L_bnd=4.214e-01 | vol=0.258 vol_eff=0.244 (/0.200) comp=3.655e+04 w=2.480e-01 | Lse=6.966e-01 Lsv=0.000e+00 rho(min/mean/max)=0.000/0.261/0.903 | rho_b=0.168 rho_v=0.210 | Δrho=2.63e-01 Δseed=6.39e-02 grad_mean=2.27e

Widget(value='<iframe src="http://localhost:40881/index.html?ui=P_0x78fe08513f10_2&reconnect=auto" class="pyvi…

threshold=0.5 (manual) | solid%=24.507%


Widget(value='<iframe src="http://localhost:40881/index.html?ui=P_0x78fe09829f90_3&reconnect=auto" class="pyvi…

(UnstructuredGrid (0x78fe0858e920)
   N Cells:    5659
   N Points:   3265
   X Bounds:   0.000e+00, 1.000e+01
   Y Bounds:   -1.274e-01, 2.101e+00
   Z Bounds:   -2.033e+00, 1.939e+00
   N Arrays:   3,
 0.5)

In [ ]:
for t in [0.25, 0.6, 0.7]:
    trainer.visualize_result_final(result, points_xyz, faces_ijk, thr=t, show_solid=False)